In [40]:

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from joblib import dump

In [41]:
df = pd.read_csv("../../data/cars.csv")
print(df.head(5))

num_cols = df.select_dtypes(include="number").columns.to_list()
cat_cols = df.select_dtypes(include="object").columns.to_list()
num_cols.remove("price_usd")
x_cols = num_cols + cat_cols
print(num_cols)
print(cat_cols)

   make_year  mileage_kmpl  engine_cc fuel_type  owner_count  price_usd  \
0       2001          8.17       4000    Petrol            4    8587.64   
1       2014         17.59       1500    Petrol            4    5943.50   
2       2023         18.09       2500    Diesel            5    9273.58   
3       2009         11.28        800    Petrol            1    6836.24   
4       2005         12.23       1000    Petrol            2    4625.79   

       brand transmission  color service_history  accidents_reported  \
0  Chevrolet       Manual  White             NaN                   0   
1      Honda       Manual  Black             NaN                   0   
2        BMW    Automatic  Black            Full                   1   
3    Hyundai       Manual   Blue            Full                   0   
4     Nissan    Automatic    Red            Full                   0   

  insurance_valid  
0              No  
1             Yes  
2             Yes  
3             Yes  
4             Ye

In [42]:
df["service_history"].fillna("Unknown" , inplace= True)

/var/folders/mm/ltg3d_2n05nb1y0p223lhx8c0000gn/T/ipykernel_84748/905142834.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["service_history"].fillna("Unknown" , inplace= True)


In [43]:
scaler = StandardScaler()
enc = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

In [44]:
X_temp ,  X_test ,y_temp, y_test = train_test_split(df[x_cols], df["price_usd"], test_size=0.2,random_state=42)
X_train,  X_val , y_train, y_val = train_test_split(df[x_cols], df["price_usd"], random_state=42, test_size= 0.1765)

In [45]:
num_pipe = Pipeline([
    #("impute", SimpleImputer(strategy = "median"), not useful cause both prod and dataset does not have missing values
    ("scale",scaler)])
cat_pipe = Pipeline([
    #("impute", SimpleImputer(strategy = "most_frequent), not useful cause both prod and dataset does not have missing values
    ("ohe", enc)
])

preprocess = ColumnTransformer([
    ("num", num_pipe , num_cols), # its important to put num first otherwise one hot columns will be scaled
    ("cat" , cat_pipe, cat_cols),
], remainder = "drop" )


In [49]:
X_train_transformed = preprocess.fit_transform(X_train)
X_val_transformed = preprocess.transform(X_val)
X_test_transformed = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()


X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_val_df   = pd.DataFrame(X_val_transformed,   columns=feature_names)
X_test_df  = pd.DataFrame(X_test_transformed,  columns=feature_names)

X_train_df.to_csv("../../data/train_features.csv", index=False)
X_val_df.to_csv("../../data/val_features.csv",   index=False)
X_test_df.to_csv("../../data/test_features.csv",  index=False)

y_train.to_csv("../../data/train_labels.csv", index=False)
y_val.to_csv("../../data/val_labels.csv",   index=False)
y_test.to_csv("../../data/test_labels.csv",  index=False)

In [48]:
dump({"preprocessor" : preprocess,
             "cat":cat_cols ,
             "num":num_cols},
            "../../models/preprocessor")

['../../models/preprocessor']